In [1]:
import time

import jax
from jax import lax
import jax.numpy as jnp

from graphax.sparse import block
from graphax.sparse.block import DenseDimension as dd, SparseDimension as sd, _dense

from itertools import product

In [2]:
from jax import tree_util
from graphax.sparse.block import BlockSparseTensor

def flatten_block_sparse_tensor(tensor):
    children = (tensor.blocks,)
    aux_data = (tensor.out_dims, tensor.primal_dims, tensor.out_shape, tensor.primal_shape, tensor.sparse_dims, tensor.pre_transforms, tensor.post_transforms)
    return children, aux_data

def unflatten_block_sparse_tensor(aux_data, children):
    blocks, = children
    out_dims, primal_dims, out_shape, primal_shape, sparse_dims, pre_transforms, post_transforms = aux_data
    return BlockSparseTensor(out_dims, primal_dims, out_shape, primal_shape, blocks, sparse_dims, pre_transforms, post_transforms)

tree_util.register_pytree_node(BlockSparseTensor, flatten_block_sparse_tensor, unflatten_block_sparse_tensor)

In [3]:
key = jax.random.PRNGKey(123)

n = 2

num_blocks, block_rows, block_cols = n, n+1, n+2
block_sparse_matrix_shape = (num_blocks*block_rows, num_blocks*block_cols)

# simple sparse representation in compressed form
blocks = jax.random.normal(key, (num_blocks, block_rows, block_cols))

out_dim = n+3
dense_matrix = jax.random.normal(key, (num_blocks*block_cols, out_dim))
print(dense_matrix.shape)

(8, 5)


In [4]:
bst = block.new_block_sparse_tensor(
    out_dims = [sd(0, num_blocks, 0, 1, block_rows)],
    primal_dims = [sd(1, num_blocks, 1, 0, block_cols)],
    blocks = blocks
)
# TODO make _dense jittable!

In [5]:
def indices(xs):
    return jnp.indices(xs).reshape(2,-1).T

indices([2,2]) # == [[0, 0], [0, 1], [1, 0], [1, 1]].T

Array([[0, 0],
       [0, 1],
       [1, 0],
       [1, 1]], dtype=int32)

In [6]:
def get_other_sparse_dim(bst, sd):
    return bst.primal_dims[sd.other_id-len(bst.out_dims)]

In [7]:
@jax.jit
def to_dense(blocks):
	"""converts a block-sparse matrix to a dense one."""
	num_blocks, block_rows, block_cols = blocks.shape
	num_rows, num_cols = num_blocks * block_rows, num_blocks * block_cols
	dense = jnp.zeros((num_rows, num_cols))

	for i in range(num_blocks):
		dense = dense.at[block_rows*i:block_rows*(i+1), block_cols*i:block_cols*(i+1)].set(blocks[i])
	return dense

block_dense_matrix = to_dense(blocks)
print(block_dense_matrix.shape)

(6, 8)


In [8]:
iters = 10

st = time.time()
for _ in range(iters):
	res = jax.jit(jnp.matmul)(block_dense_matrix, dense_matrix)
print(f'Dense (sparse @ dense) matmul time: {time.time() - st}')

Dense (sparse @ dense) matmul time: 0.010147333145141602


In [9]:
# use reshape and einsum
@jax.jit
def sparse_mul(blocks, dense):
    num_blocks, block_rows, block_cols = blocks.shape
    out_dim = dense.shape[-1]
    print(blocks.shape, dense.shape)
    dense = dense.reshape(num_blocks, block_cols, -1)
    print(blocks.shape, dense.shape)
    res = lax.dot(blocks, dense, dimension_numbers=(([2], [1]), ([0], [0])))
    
    print(res.shape)
    return res.reshape(-1, out_dim)

print(jax.make_jaxpr(sparse_mul)(blocks, dense_matrix))
sparse_res = sparse_mul(blocks, dense_matrix)
# check if results are equal
print(res[:2, :2], sparse_res[:2, :2], sep='\n')
jnp.allclose(res, sparse_res)

(2, 3, 4) (8, 5)
(2, 3, 4) (2, 4, 5)
(2, 3, 5)
{ lambda ; a:f32[2,3,4] b:f32[8,5]. let
    c:f32[6,5] = jit[
      name=sparse_mul
      jaxpr={ lambda ; a:f32[2,3,4] b:f32[8,5]. let
          d:f32[2,4,5] = reshape[
            dimensions=None
            new_sizes=(2, 4, 5)
            sharding=None
          ] b
          e:f32[2,3,5] = dot_general[dimension_numbers=(([2], [1]), ([0], [0]))] a
            d
          c:f32[6,5] = reshape[dimensions=None new_sizes=(6, 5) sharding=None] e
        in (c,) }
    ] a b
  in (c,) }
[[ 2.625091    0.95891714]
 [-2.30002     0.8644688 ]]
[[ 2.625091    0.95891726]
 [-2.30002     0.8644688 ]]


Array(True, dtype=bool)

In [10]:
st = time.time()
for i in range(iters):
	res = sparse_mul(blocks, dense_matrix)
print(f'Sparse (sparse @ dense) matmul time: {time.time() - st}')

Sparse (sparse @ dense) matmul time: 0.0011057853698730469


In [11]:
# use reshape and einsum
@jax.jit
def sparse_mul(blocks: block.BlockSparseTensor, dense):
    block_nums = blocks.blocks.shape[:blocks.sparse_dims]
    block_sizes = [d.block_size for d in blocks.primal_dims if isinstance(d, sd)]
    block_idxs = tuple(range(blocks.sparse_dims))
    val_dims = [d.val_dim+blocks.sparse_dims for d in blocks.primal_dims if isinstance(d, sd)]
    print(blocks.shape, dense.shape)
    dense = dense.reshape(*block_nums, *block_sizes, -1)
    print(blocks.blocks.shape, dense.shape)
    print(val_dims, block_nums)
    res = lax.dot(blocks.blocks, dense, dimension_numbers=((val_dims, [x-1 for x in val_dims]), (block_idxs,)*2))
    
    print(res.shape)
    print(res.reshape(-1, out_dim).shape)
    return res.reshape(blocks.out_shape + dense.shape[-blocks.sparse_dims:])

print(jax.make_jaxpr(sparse_mul)(bst, dense_matrix))
sparse_res = sparse_mul(bst, dense_matrix)
# check if results are equal
print(res[:2, :2], sparse_res[:2, :2], sep='\n')
jnp.allclose(res, sparse_res)

(6, 8) (8, 5)
(2, 3, 4) (2, 4, 5)
[2] (2,)
(2, 3, 5)
(6, 5)
{ lambda ; a:f32[2,3,4] b:f32[8,5]. let
    c:f32[6,5] = jit[
      name=sparse_mul
      jaxpr={ lambda ; a:f32[2,3,4] b:f32[8,5]. let
          d:f32[2,4,5] = reshape[
            dimensions=None
            new_sizes=(2, 4, 5)
            sharding=None
          ] b
          e:f32[2,3,5] = dot_general[dimension_numbers=(([2], [1]), ([0], [0]))] a
            d
          _:f32[6,5] = reshape[dimensions=None new_sizes=(6, 5) sharding=None] e
          c:f32[6,5] = reshape[dimensions=None new_sizes=(6, 5) sharding=None] e
        in (c,) }
    ] a b
  in (c,) }
[[ 2.625091    0.95891726]
 [-2.30002     0.8644688 ]]
[[ 2.625091    0.95891726]
 [-2.30002     0.8644688 ]]


Array(True, dtype=bool)

In [12]:
st = time.time()
for i in range(iters):
	res = sparse_mul(bst, dense_matrix)
print(f'Super Sparse (sparse @ dense) matmul time: {time.time() - st}')

Super Sparse (sparse @ dense) matmul time: 0.0012731552124023438


In [13]:
st = time.time()
for _ in range(iters):
	res = jax.jit(jnp.matmul)(dense_matrix.T, block_dense_matrix.T)
print(f'Dense (dense @ sparse) matmul time: {time.time() - st}')

Dense (dense @ sparse) matmul time: 0.05196070671081543


In [14]:
# use reshape and einsum
@jax.jit
def sparse_mul(dense, blocks):
    num_blocks, block_rows, block_cols = blocks.shape
    out_dim = dense.shape[-1]
    print(dense.shape, blocks.shape)
    dense = dense.reshape(num_blocks, block_cols, -1)
    blocks = blocks.transpose(0,2,1)
    print(dense.shape, blocks.shape)
    res = lax.dot(blocks, dense, dimension_numbers=(([1], [1]), ([0], [0])))
    
    print(res.shape)
    return res.reshape(-1, out_dim).transpose(1,0)

print(jax.make_jaxpr(sparse_mul)(dense_matrix, blocks))
sparse_res = sparse_mul(dense_matrix, blocks)
# check if results are equal
print(res[:2, :2], sparse_res[:2, :2], sep='\n')
jnp.allclose(res, sparse_res)

(8, 5) (2, 3, 4)
(2, 4, 5) (2, 4, 3)
(2, 3, 5)
{ lambda ; a:f32[8,5] b:f32[2,3,4]. let
    c:f32[5,6] = jit[
      name=sparse_mul
      jaxpr={ lambda ; a:f32[8,5] b:f32[2,3,4]. let
          d:f32[2,4,5] = reshape[
            dimensions=None
            new_sizes=(2, 4, 5)
            sharding=None
          ] a
          e:f32[2,4,3] = transpose[permutation=(0, 2, 1)] b
          f:f32[2,3,5] = dot_general[dimension_numbers=(([1], [1]), ([0], [0]))] e
            d
          g:f32[6,5] = reshape[dimensions=None new_sizes=(6, 5) sharding=None] f
          c:f32[5,6] = transpose[permutation=(1, 0)] g
        in (c,) }
    ] a b
  in (c,) }
[[ 2.625091   -2.30002   ]
 [ 0.95891714  0.8644688 ]]
[[ 2.625091   -2.30002   ]
 [ 0.95891726  0.8644688 ]]


Array(True, dtype=bool)

In [15]:
st = time.time()
for i in range(iters):
	res = sparse_mul(dense_matrix, blocks)
print(f'Sparse (dense @ sparse) matmul time: {time.time() - st}')

Sparse (dense @ sparse) matmul time: 0.0010139942169189453


In [16]:
# use reshape and einsum
@jax.jit
def sparse_mul(dense, blocks: block.BlockSparseTensor):
    block_nums = blocks.blocks.shape[:blocks.sparse_dims]
    block_sizes = [d.block_size for d in blocks.primal_dims if isinstance(d, sd)]
    block_idxs = tuple(range(blocks.sparse_dims))
    val_dims = [d.val_dim+blocks.sparse_dims for d in blocks.primal_dims if isinstance(d, sd)]
    print(blocks.shape, dense.shape)
    dense = dense.reshape(*block_nums, *block_sizes, -1)
    transposed_axes = [blocks.primal_dims[d.other_id-len(blocks.out_dims)].val_dim for d in blocks.out_dims] \
                        + [blocks.out_dims[d.other_id].val_dim for d in blocks.primal_dims]
    print(transposed_axes)
    blocks.blocks = blocks.blocks.transpose(0,*[x+1 for x in transposed_axes])
    print(blocks.blocks.shape, dense.shape)
    print(val_dims, block_nums)
    res = lax.dot(blocks.blocks, dense, dimension_numbers=(([x-1 for x in val_dims],)*2, (block_idxs,)*2))
    
    print(res.shape)
    print(-1, out_dim)
    print(dense.shape[:blocks.sparse_dims] + blocks.primal_shape)
    print(blocks.out_shape + dense.shape[-blocks.sparse_dims:])
    return res.reshape(blocks.out_shape + dense.shape[-blocks.sparse_dims:]).transpose(transposed_axes)

print(jax.make_jaxpr(sparse_mul)(dense_matrix, bst))
sparse_res = sparse_mul(dense_matrix, bst)
# check if results are equal
print(res[:2, :2], sparse_res[:2, :2], sep='\n')
jnp.allclose(res, sparse_res)

(6, 8) (8, 5)
[1, 0]
(2, 4, 3) (2, 4, 5)
[2] (2,)
(2, 3, 5)
-1 5
(2, 8)
(6, 5)
{ lambda ; a:f32[8,5] b:f32[2,3,4]. let
    c:f32[5,6] = jit[
      name=sparse_mul
      jaxpr={ lambda ; a:f32[8,5] b:f32[2,3,4]. let
          d:f32[2,4,5] = reshape[
            dimensions=None
            new_sizes=(2, 4, 5)
            sharding=None
          ] a
          e:f32[2,4,3] = transpose[permutation=(0, 2, 1)] b
          f:f32[2,3,5] = dot_general[dimension_numbers=(([1], [1]), ([0], [0]))] e
            d
          g:f32[6,5] = reshape[dimensions=None new_sizes=(6, 5) sharding=None] f
          c:f32[5,6] = transpose[permutation=(1, 0)] g
        in (c,) }
    ] a b
  in (c,) }
[[ 2.625091   -2.30002   ]
 [ 0.95891726  0.8644688 ]]
[[ 2.625091   -2.30002   ]
 [ 0.95891726  0.8644688 ]]


Array(True, dtype=bool)

In [17]:
st = time.time()
for i in range(iters):
	res = sparse_mul(dense_matrix, bst)
print(f'Super Sparse (dense @ sparse) matmul time: {time.time() - st}')

Super Sparse (dense @ sparse) matmul time: 0.0005824565887451172
